# Building a sentence embedder from scratch with ONNX Runtime

The usual way to turn text into an embedding in Python is:

```python
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
v = model.encode("Can I still join the course?")   # -> array of 384 floats
```

That one line hides a lot, and it pulls in **PyTorch** (~1–2 GB). In this notebook we rebuild `encode()` by hand,
one step at a time, using only:

| library | what it does | size |
|---|---|---|
| `tokenizers` | turns text into integer ids | small |
| `onnxruntime` | runs the neural network | ~15 MB |
| `numpy` | the maths we do ourselves | – |

By the end you'll have an `OnnxEmbedder` class whose outputs are **identical** to SentenceTransformer's,
and you'll know exactly what every line does.

## Setup

```bash
pip install -r requirements.txt                 # numpy, onnxruntime, tokenizers, huggingface-hub, onnx
pip install sentence-transformers               # OPTIONAL: only for the "did we match?" checks (pulls in PyTorch)
```

Everything runs on a normal CPU. The model (~90 MB) is downloaded from Hugging Face on first run, and Step 8
downloads a public FAQ dataset (~1,500 short documents) to search over.

**Who this is for:** you know basic Python. No NumPy, PyTorch or transformer knowledge needed; the notebook
explains each piece as it comes up.

## The big picture

```
 "Can I still join?"
        │
        ▼
 ┌──────────────┐   STEP 1  TOKENIZE
 │  tokenizer   │   text  ──►  [101, 2064, 1045, 2145, 3693, 1029, 102]
 └──────────────┘
        │
        ▼
 ┌──────────────┐   STEP 2  RUN THE TRANSFORMER  (the only "AI" part)
 │  ONNX model  │   7 token ids  ──►  7 vectors of 384 numbers   shape (7, 384)
 └──────────────┘
        │
        ▼
 ┌──────────────┐   STEP 3  MEAN POOLING
 │   average    │   7 vectors  ──►  1 vector                    shape (384,)
 └──────────────┘
        │
        ▼
 ┌──────────────┐   STEP 4  NORMALIZE
 │ length = 1   │   so that  a · b  ==  cosine similarity
 └──────────────┘
        │
        ▼
   embedding (384,)
```

Only step 2 uses the trained network. Steps 1, 3 and 4 are plain code, and that's the code
sentence-transformers was writing for us.

---
# Step 0: Download the model files

Models on Hugging Face are git repos full of files. The `all-MiniLM-L6-v2` repo contains:

- `onnx/model.onnx`: the network exported to ONNX format (the thing we'll run)
- `tokenizer.json`: the vocabulary and the rules for splitting text
- `modules.json`, `1_Pooling/config.json`, `sentence_bert_config.json`: the **recipe** sentence-transformers follows
  around the network. We read these so that we copy the recipe exactly instead of guessing.

`hf_hub_download` downloads a file once, caches it in `~/.cache/huggingface`, and returns the local path.

In [ ]:
import os
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"   # silence a harmless Windows warning

import json
import numpy as np
from huggingface_hub import hf_hub_download

REPO = "sentence-transformers/all-MiniLM-L6-v2"

model_path     = hf_hub_download(REPO, "onnx/model.onnx")
tokenizer_path = hf_hub_download(REPO, "tokenizer.json")

print("model file:", model_path)
print("size: %.1f MB" % (os.path.getsize(model_path) / 1e6))

### Read the recipe

`modules.json` lists the stages sentence-transformers runs, in order:

In [ ]:
modules = json.load(open(hf_hub_download(REPO, "modules.json")))
for m in modules:
    print(m["idx"], m["type"])

**Transformer → Pooling → Normalize.** These are steps 2, 3 and 4 from the diagram. (Step 1, tokenizing, happens
inside the Transformer module.)

Next, *which kind* of pooling, and the maximum text length:

In [ ]:
pooling_cfg = json.load(open(hf_hub_download(REPO, "1_Pooling/config.json")))
st_cfg      = json.load(open(hf_hub_download(REPO, "sentence_bert_config.json")))

print(json.dumps(pooling_cfg, indent=2))
print(st_cfg)

MAX_LENGTH = st_cfg["max_seq_length"]
EMBED_DIM  = pooling_cfg["word_embedding_dimension"]

What we learned:

- `pooling_mode_mean_tokens: true`: step 3 is a plain **average** of the token vectors (not "take the first token" or "take the max").
- `word_embedding_dimension: 384`: every vector has 384 numbers.
- `max_seq_length: 256`: texts longer than 256 tokens get cut off.

Now we have everything we need to rebuild the pipeline.

---
# Before we start: a NumPy crash course

Every step after tokenizing works with **NumPy arrays**. If you haven't used NumPy, this section teaches exactly
the parts this notebook needs, nothing more. Each part is referenced later with a 🔎 **NumPy note**.

`np` is just the conventional short name: `import numpy as np`.

## N.1 Why NumPy at all?

Three reasons, from most to least important for us:

1. **ONNX Runtime speaks NumPy.** `session.run()` only accepts NumPy arrays as input and returns NumPy arrays
   as output. PyTorch has its own array type (`torch.Tensor`); without PyTorch, NumPy is the standard.
2. **Speed.** A Python list stores each number as a separate Python object scattered around memory. A NumPy array
   stores raw numbers side by side in one block, and its operations run as compiled C loops instead of Python loops.
3. **Less code.** `a * b` multiplies whole arrays at once. No `for` loops.

Let's measure #2 with the exact job from Step 8: scoring a query against ~1,500 document vectors of 384 numbers each.

In [ ]:
import time

rng = np.random.default_rng(0)
docs_np  = rng.standard_normal((1489, 384)).astype(np.float32)   # a fake "document matrix"
query_np = rng.standard_normal(384).astype(np.float32)

docs_list  = docs_np.tolist()          # the same numbers as plain Python lists
query_list = query_np.tolist()

# pure Python: two nested loops
start = time.perf_counter()
scores_py = [sum(d_i * q_i for d_i, q_i in zip(doc, query_list)) for doc in docs_list]
t_py = time.perf_counter() - start

# NumPy: one expression
start = time.perf_counter()
scores_np = docs_np @ query_np
t_np = time.perf_counter() - start

print(f"pure Python: {t_py * 1000:8.1f} ms")
print(f"NumPy:       {t_np * 1000:8.3f} ms   (~{t_py / t_np:.0f}x faster)")
print("same answer:", np.allclose(scores_py, scores_np, atol=1e-3))

Same result, and NumPy is dozens of times faster here (the gap grows with bigger arrays). That's why every ML library is built on arrays.

## N.2 Creating arrays: `shape` and `dtype`

Every array has two key properties:

- **`shape`**: how many numbers along each direction (a tuple)
- **`dtype`**: what type each number is (all numbers in one array share a single type)

In [ ]:
a = np.array([1, 2, 3])                    # from a list   -> 1-D
b = np.array([[1, 2, 3],
              [4, 5, 6]])                  # from a list of lists -> 2-D

print("a:", a, " shape:", a.shape, " dtype:", a.dtype)
print("b:\n", b)
print("b shape:", b.shape, "  <- (rows, columns)")
print("b.ndim: ", b.ndim,  "      <- number of dimensions ('axes')")

### Why dtype matters

| dtype | bytes per number | used in this notebook for |
|---|---|---|
| `int64` | 8 | token ids and masks (**ONNX demands int64**, see Step 2) |
| `float32` | 4 | embeddings (what the model outputs) |
| `float64` | 8 | NumPy's default for decimals; twice the memory of float32 |

You choose it with `dtype=` when creating, or convert with `.astype(...)`:

In [ ]:
c = np.array([1, 2, 3], dtype=np.int64)
print(c.dtype, "->", c.astype(np.float32).dtype, c.astype(np.float32))

big = np.zeros((1489, 384))                      # float64 by default
print(f"1489 x 384 as float64: {big.nbytes / 1e6:.1f} MB,  as float32: {big.astype(np.float32).nbytes / 1e6:.1f} MB")

## N.3 Thinking in shapes

This is the most useful mental model in the whole notebook. **Read a shape from left to right as "for each … for each …":**

```
shape (384,)          one vector                       an embedding
shape (2, 384)        for each of 2 texts: a vector     a batch of embeddings      (a table: 2 rows, 384 columns)
shape (2, 13, 384)    for each of 2 texts,
                        for each of 13 tokens: a vector  the transformer's output  (a stack of 2 tables)
```

Each position in the shape is called an **axis**: axis 0 is the first number, axis 1 the second, and so on.
When the notebook says "average over the tokens", that means "average over **axis 1**" of a `(batch, tokens, 384)` array.

In [ ]:
x = np.arange(24).reshape(2, 3, 4)     # numbers 0..23 arranged as 2 tables of 3 rows x 4 columns
print("shape:", x.shape)
print(x)

## N.4 Indexing and slicing: picking parts of an array

Inside `[ ]`, give one position per axis, separated by commas. `:` means "everything along this axis", and
`a:b` means "from a up to (not including) b".

In [ ]:
print("x[0]        -> first table, shape", x[0].shape)
print(x[0])
print()
print("x[0, 1]     -> table 0, row 1:", x[0, 1])
print("x[0, 1, 2]  -> one number:    ", x[0, 1, 2])
print("x[0, 1, :2] -> first 2 of it: ", x[0, 1, :2])
print()
print("x[:, 0]     -> row 0 of EVERY table, shape", x[:, 0].shape)
print(x[:, 0])

In the notebook:

- `token_embeddings[0, 0, :5]` = text 0, token 0 (`[CLS]`), first 5 of its 384 numbers
- `token_embeddings[:, 0]` = the `[CLS]` vector of every text (that's "CLS pooling", exercise 3)
- `texts[i:i + 32]` = the next batch of 32 (slicing works on Python lists too, and going past the end is fine)

You can also index with a **list of positions** ("fancy indexing"). It picks those items in that order:

In [ ]:
scores_demo = np.array([0.1, 0.9, 0.4, 0.7])
print(scores_demo[[1, 3]])       # -> items 1 and 3

## N.5 Element-wise maths: no loops

`+ - * /` between two arrays of the same shape work **number by number**. With a single number, it applies to all of them.

In [ ]:
p = np.array([1.0, 2.0, 3.0])
q = np.array([10.0, 20.0, 30.0])
print("p + q =", p + q)
print("p * q =", p * q)        # NOT a dot product: three separate multiplications
print("p / 2 =", p / 2)
print("p ** 2 =", p ** 2)

## N.6 Summing and averaging along an axis

`.sum()`, `.mean()`, `.max()` collapse numbers. **`axis=` says which direction gets collapsed** (that axis disappears from the shape):

In [ ]:
t = np.array([[1, 2, 3],
              [4, 5, 6]])            # shape (2, 3)

print("t.sum()       =", t.sum(), "            (everything)")
print("t.sum(axis=0) =", t.sum(axis=0), "  shape", t.sum(axis=0).shape, " (down the columns: rows collapse)")
print("t.sum(axis=1) =", t.sum(axis=1), "     shape", t.sum(axis=1).shape, "   (across each row: columns collapse)")
print()
print("keepdims=True keeps the collapsed axis with size 1:")
print("t.sum(axis=1, keepdims=True) shape:", t.sum(axis=1, keepdims=True).shape)
print(t.sum(axis=1, keepdims=True))

On our 3-D transformer output `(batch, tokens, 384)`, `.sum(axis=1)` adds up the token vectors for each text,
giving `(batch, 384)`. That's the core of mean pooling (Step 3).

## N.7 Broadcasting: when shapes don't match

NumPy can combine arrays of **different** shapes by "stretching" axes of size 1. The rule:

> Line the shapes up **from the right**. Each pair of sizes must be **equal**, or one of them must be **1**
> (the 1 gets stretched to match). A missing axis on the left counts as 1.

```
(2, 3)  *  (3,)      ->  (2, 3)  *  (1, 3)   ✅  the row is reused for every row
(2, 3)  *  (2, 1)    ->  ✅  the column is reused for every column
(2, 3)  *  (2,)      ->  compare 3 vs 2       ❌  error
```

In [ ]:
t = np.array([[1.0, 2.0, 3.0],
              [4.0, 5.0, 6.0]])            # (2, 3)

row    = np.array([10.0, 100.0, 1000.0])   # (3,)
column = np.array([[1.0], [0.0]])          # (2, 1)

print("t * row    (each ROW multiplied by [10, 100, 1000]):\n", t * row)
print("t * column (row 0 kept, row 1 zeroed):\n", t * column)

try:
    t * np.array([1.0, 0.0])               # (2,) lines up with the 3 -> mismatch
except ValueError as e:
    print("\nt * (2,) fails:", e)

Look at `t * column`: multiplying by a column of 1s and 0s **keeps or zeroes whole rows**. That is exactly
how we'll remove padding tokens in Step 3.

### Adding an axis of size 1: `[..., None]`

Often we need to turn shape `(2, 13)` into `(2, 13, 1)` so it broadcasts against `(2, 13, 384)`.
Indexing with `None` inserts a new axis of size 1. `...` means "all the existing axes":

In [ ]:
mask_demo = np.array([[1, 1, 0],
                      [1, 0, 0]])            # (2, 3)

print("mask_demo.shape            :", mask_demo.shape)
print("mask_demo[..., None].shape :", mask_demo[..., None].shape, "  <- new axis at the end")
print("mask_demo[:, None].shape   :", mask_demo[:, None].shape,   "  <- new axis in the middle")

`mask[..., None]` is the same as `mask.reshape(2, 3, 1)` or `np.expand_dims(mask, -1)`; it's just the shortest way to write it.

## N.8 Dot product and matrix multiplication: `@`

The **dot product** of two vectors multiplies pair by pair and sums: `[1,2,3] · [4,5,6] = 1·4 + 2·5 + 3·6 = 32`.
In NumPy that's `a @ b` (or `a.dot(b)`).

With a **matrix** on the left, `@` does one dot product **per row**. That's how one line scores a query against
every document:

In [ ]:
a = np.array([1.0, 2.0, 3.0])
b = np.array([4.0, 5.0, 6.0])
print("a @ b =", a @ b, "  (same as (a * b).sum() =", (a * b).sum(), ")")

D = np.array([[1.0, 0.0, 0.0],      # "document" 0
              [0.0, 1.0, 0.0],      # "document" 1
              [1.0, 1.0, 1.0]])     # "document" 2   -> shape (3 docs, 3 numbers)
print("D @ a =", D @ a, "  <- one score per document, shape", (D @ a).shape)

Shape rule for `@`: `(n, k) @ (k,) -> (n,)` and `(n, k) @ (k, m) -> (n, m)`. The inner sizes (`k`) must match.
In Step 8: `(n_docs, 384) @ (384,) -> (n_docs,)`, one similarity score per FAQ document.

## N.9 The handful of functions we use

In [ ]:
v = np.array([3.0, 4.0])
s = np.array([0.2, 0.9, 0.1, 0.7])

print("np.linalg.norm(v)  =", np.linalg.norm(v), "     length of a vector: sqrt(3² + 4²)")
print("np.argmax(s)       =", np.argmax(s), "       POSITION of the biggest value")
print("np.argsort(s)      =", np.argsort(s), " positions that would sort s, smallest first")
print("np.argsort(-s)[:2] =", np.argsort(-s)[:2], "     positions of the 2 biggest (negate = sort descending)")
print("np.clip(s, 0.15, 0.8) =", np.clip(s, 0.15, 0.8), "  force values into a range")
print("np.zeros_like(s)   =", np.zeros_like(s), " zeros with the same shape + dtype")
print("np.vstack([[1, 2], [3, 4]]) ->\n", np.vstack([np.array([[1, 2]]), np.array([[3, 4]])]), " stack blocks of rows into one array")
print("np.abs(-s).max()   =", np.abs(-s).max(), "  biggest absolute value")

### Cheat sheet

| you see | it means | where |
|---|---|---|
| `np.array(list, dtype=np.int64)` | make an array of 64-bit integers | Step 1.5 |
| `x.shape` | size along each axis | everywhere |
| `x[0, 0, :5]` | text 0, token 0, first 5 numbers | Step 2.2 |
| `x * mask` | element-wise multiply, with broadcasting | Step 3 |
| `mask[..., None]` | add a size-1 axis at the end | Step 3 |
| `x.sum(axis=1)` | add up along axis 1 (the tokens) | Step 3 |
| `np.linalg.norm(x, axis=1, keepdims=True)` | length of each row, kept as a column | Step 4 |
| `a @ b` | dot product / matrix multiply | Steps 4, 8 |
| `np.vstack(chunks)` | glue batch results together | Step 6 |
| `np.argsort(-scores)[:5]` | positions of the top 5 scores | Step 8 |

Don't memorize it. Come back here whenever a 🔎 note points you to a section.

---
# Step 1: Tokenization (text ➜ integers)

A neural network only does arithmetic on numbers, so the text has to become numbers first.

The tokenizer has a fixed **vocabulary**: a dictionary from word pieces to integer ids.

In [ ]:
from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file(tokenizer_path)

print("vocabulary size:", tokenizer.get_vocab_size())
for word in ["course", "join", "docker", "zoomcamp", "[CLS]", "[SEP]", "[PAD]", "[UNK]"]:
    print(f"{word!r:12} -> {tokenizer.token_to_id(word)}")

There are 30,522 entries. Note that `"course"` has an id, but `"docker"` and `"zoomcamp"` return `None`:
they aren't in the vocabulary as whole words. We'll see how the tokenizer copes with that in a moment.

### 1.0 Gotcha: the tokenizer file comes with settings baked in

`tokenizer.json` doesn't only hold the vocabulary. It can also store padding and truncation settings. Let's look:

In [ ]:
print("truncation:", tokenizer.truncation)
print("padding:   ", tokenizer.padding)

This file says **"cut at 128 tokens, and always pad to exactly 128"**. But `sentence_bert_config.json` said
`max_seq_length: 256`, and sentence-transformers quietly overrides these settings when it loads the model.

If we kept the file's defaults, long texts would get cut at 128 instead of 256, and our embeddings would **not**
match SentenceTransformer's. Hidden defaults like this are the classic bug when you swap libraries.

So we switch both off, then turn them back on ourselves, deliberately, in 1.3 and 1.4:

In [ ]:
tokenizer.no_padding()
tokenizer.no_truncation()
print("truncation:", tokenizer.truncation)
print("padding:   ", tokenizer.padding)

### 1.1 Encode one sentence

In [ ]:
enc = tokenizer.encode("Can I still join the course after the start date?")

print(f"{'token':>10} | id")
print("-" * 18)
for tok, id_ in zip(enc.tokens, enc.ids):
    print(f"{tok:>10} | {id_}")

Three things to notice:

1. **Lowercased**: `Can` became `can`. This model is "uncased", so `Docker` and `docker` are the same to it.
2. **`[CLS]` (101) at the start and `[SEP]` (102) at the end** were added automatically. BERT-family models were
   trained with these markers, so we must always include them.
3. The `?` is its own token.

### 1.2 Unknown words get split into sub-words

What happens to `docker` and `zoomcamp`, which aren't in the vocabulary?

In [ ]:
for text in ["docker", "zoomcamp", "pgvector", "embeddings"]:
    print(f"{text:12} -> {tokenizer.encode(text).tokens}")

This is **WordPiece**. Rare words are broken into known pieces, and `##` means "glued to the previous piece".
So the model never sees a truly unknown word, only combinations of pieces it learned during training.

### 1.3 Truncation: long texts get cut

The model handles at most 256 tokens here (its position table only goes so far, and attention cost grows with the
square of the length), so we tell the tokenizer to cut anything longer:

In [ ]:
long_text = "course " * 1000

print("without truncation:", len(tokenizer.encode(long_text).ids), "tokens")

tokenizer.enable_truncation(max_length=MAX_LENGTH)
print("with truncation:   ", len(tokenizer.encode(long_text).ids), "tokens")

> ⚠️ Anything after token 256 is **silently ignored**. For long FAQ answers this means the end of the text
> doesn't affect the embedding. That's one reason real RAG systems **chunk** long documents before embedding them.

### 1.4 Batches need padding (and an attention mask)

For speed we embed many texts at once. A batch has to be a rectangle (every row the same length), so
shorter texts get filled up with `[PAD]` tokens.

With no `length=` argument, `enable_padding` pads to the **longest text in the batch**. That's less wasteful
than the file's "always pad to 128".

In [ ]:
tokenizer.enable_padding(pad_id=0, pad_token="[PAD]")

batch = tokenizer.encode_batch(["Hi", "How to install Docker on Windows?"])

for e in batch:
    print("tokens:        ", e.tokens)
    print("ids:           ", e.ids)
    print("attention_mask:", e.attention_mask)
    print()

`attention_mask` is the key idea here:

- `1` = real token, pay attention to it
- `0` = padding, it's only there to make the rectangle, ignore it

The model uses the mask internally, and **we will need it again in step 3**. Keep it in mind.

### 1.5 Turn it into numpy arrays

ONNX Runtime wants numpy arrays of `int64`. We also need `token_type_ids`, a leftover from BERT's training where
two sentences were fed at once (sentence A = 0, sentence B = 1). We only ever feed one text, so it's all zeros.

In [ ]:
def tokenize(texts):
    encodings = tokenizer.encode_batch(texts)
    input_ids      = np.array([e.ids            for e in encodings], dtype=np.int64)
    attention_mask = np.array([e.attention_mask for e in encodings], dtype=np.int64)
    token_type_ids = np.zeros_like(input_ids)
    return input_ids, attention_mask, token_type_ids

texts = [
    "Can I still join the course after the start date?",
    "How to install Docker on Windows?",
]
input_ids, attention_mask, token_type_ids = tokenize(texts)

print("input_ids shape:", input_ids.shape, " <- (batch_size, sequence_length)")
print(input_ids)
print(attention_mask)

> 🔎 **NumPy note (N.2, N.3).** `np.array([e.ids for e in encodings])` turns a *list of lists* into a 2-D array:
> one row per text, one column per token position. That only works because padding made every row the **same
> length**. A grid can't have rows of different lengths. Without padding, NumPy refuses:

In [ ]:
try:
    np.array([[101, 7632, 102], [101, 2129, 2000, 16500, 102]], dtype=np.int64)   # rows of length 3 and 5
except ValueError as e:
    print("ValueError:", e)

> `np.zeros_like(input_ids)` makes an array of zeros with the same shape `(2, 13)` and dtype `int64`. That's
> exactly what `token_type_ids` needs to be.

✅ **Step 1 done.** Two texts are now a `(2, 13)` grid of integers plus a matching mask.
The second text is shorter, so its row ends in zeros (padding).

---
# Step 2: Run the transformer with ONNX Runtime

### What is an `.onnx` file?

A neural network is two things:

1. a **graph** of operations (matrix multiply, add, softmax, …) that says how numbers flow through it
2. the **weights**: millions of numbers learned during training

PyTorch keeps both in Python objects. **ONNX** (Open Neural Network Exchange) saves both into one portable file.
**ONNX Runtime** is a small, fast C++ engine that loads that file and executes the graph. It doesn't need
PyTorch or Python training code, only the file.

### How ONNX lets us drop PyTorch

PyTorch is only needed **once**, by whoever *exports* the model. After that, anyone can run the file without it:

```
 EXPORT (done once, needs PyTorch, usually by the model publisher)
 ─────────────────────────────────────────────────────────────────
   PyTorch model  (Python classes + trained weights)
         │
         │  torch.onnx.export(model, example_input)
         │    -> runs the model ONCE on an example input and records every
         │       math operation it performs, in order, as a graph
         ▼
   model.onnx  =  graph of operations  +  the weights, in one file


 INFERENCE (what we do, no PyTorch at all)
 ──────────────────────────────────────────
   ort.InferenceSession("model.onnx")
         1. load the graph and weights
         2. OPTIMIZE the graph (merge small ops into bigger, faster ones)
         3. pick the fastest implementation of each op for your hardware
            (the "execution provider": CPU, CUDA GPU, DirectML, CoreML, ...)
         │
         ▼
   session.run(inputs)  ->  outputs
```

An analogy: PyTorch is the kitchen where a recipe was invented. `model.onnx` is the recipe written down, ingredients
included. ONNX Runtime is a cook who can follow any written recipe without needing that kitchen.

So here's what we replace, piece by piece:

| SentenceTransformer does… | using… | we do it with… |
|---|---|---|
| tokenize | `transformers` tokenizer (which calls `tokenizers` underneath) | `tokenizers` directly (Step 1) |
| run the 6-layer network | **PyTorch** (`torch.nn` modules, executed op by op from Python) | **`onnxruntime` + `model.onnx`** (Step 2) |
| mean pooling | torch tensor ops | numpy (Step 3) |
| normalize | torch tensor ops | numpy (Step 4) |

### 2.0 Look inside `model.onnx`

The `onnx` package (a dev-only dependency, not needed for inference) can open the file so we can see what was recorded.

In [ ]:
import onnx
from collections import Counter

onnx_model = onnx.load(model_path)
graph = onnx_model.graph

print("exported by:     ", onnx_model.producer_name, onnx_model.producer_version)
print("ONNX opset:      ", onnx_model.opset_import[0].version)
print("operations:      ", len(graph.node))
print("weight tensors:  ", len(graph.initializer))
n_params = sum(int(np.prod(t.dims)) for t in graph.initializer)
print(f"total parameters: {n_params / 1e6:.1f} million")

- **`exported by: pytorch`**: proof of the story above. Someone ran `torch.onnx.export` on the PyTorch model and
  saved the result. We get the benefit without installing PyTorch.
- **opset**: the version of the ONNX operator set the file uses (like a language version). A runtime must
  support at least that opset. Any recent `onnxruntime` does.
- **~22.6 million parameters**: that's what "MiniLM" means. It's small. (Big LLMs have billions.)

### What kinds of operations are in the graph?

In [ ]:
op_counts = Counter(node.op_type for node in graph.node)
for op, n in op_counts.most_common(15):
    print(f"{op:12} {n}")

This *is* the transformer, written out as basic maths:

- **`MatMul`** (matrix multiply) does the real work. Each of the 6 layers has 6 weight multiplications
  (query, key, value, attention output, and the two feed-forward layers), plus 2 that combine queries/keys/values.
  That's 6 × 8 = 48.
- **`ReduceMean`, `Sub`, `Sqrt`, `Div`** are **LayerNorm**, written out step by step (mean, variance, square root, divide).
- **`Gather`** is a table lookup. That's how token ids become vectors (next cell).
- **`Shape`, `Unsqueeze`, `Reshape`, `Constant`** are bookkeeping for array shapes. ONNX Runtime removes many of
  these when it optimizes (section 2.5).

Let's confirm the layer count and the 36 weight multiplications:

In [ ]:
import re

weight_names = {t.name for t in graph.initializer}
layers = sorted({int(m) for node in graph.node for m in re.findall(r"layer\.(\d+)", node.name)})
matmul_with_weights = sum(1 for n in graph.node if n.op_type == "MatMul" and any(i in weight_names for i in n.input))

print("transformer layers:", layers)
print("MatMuls that use a weight matrix:", matmul_with_weights)

### The biggest weights

`graph.initializer` holds the weights. The largest ones:

In [ ]:
for t in sorted(graph.initializer, key=lambda t: -int(np.prod(t.dims)))[:5]:
    print(f"{t.name:40} shape={list(t.dims)}")

The biggest one, `embeddings.word_embeddings.weight`, has shape **(30522, 384)**: one 384-number row **for every
token in the vocabulary** from Step 1. So the very first thing the network does with `input_ids` is look up rows in
this table (`Gather`).

We can do that lookup ourselves:

In [ ]:
from onnx import numpy_helper

word_table = numpy_helper.to_array(next(t for t in graph.initializer if t.name == "embeddings.word_embeddings.weight"))
print("table shape:", word_table.shape)

course_id = tokenizer.token_to_id("course")
print(f"'course' is id {course_id}, so its starting vector is row {course_id}:")
print(word_table[course_id][:5], "...")

That row is the **same for every sentence**, before the network has seen any context. The 6 layers after the lookup
mix in the surrounding words. That's why, in 2.3 below, the same token ends up with different vectors in different
sentences.

Now let's run the graph.

### 2.1 Load the model and ask what it expects

In [ ]:
import onnxruntime as ort

session = ort.InferenceSession(model_path, providers=["CPUExecutionProvider"])

print("INPUTS")
for i in session.get_inputs():
    print(f"  {i.name:16} shape={i.shape}  type={i.type}")

print("OUTPUTS")
for o in session.get_outputs():
    print(f"  {o.name:16} shape={o.shape}  type={o.type}")

That's the model's "API contract":

- It takes exactly the three arrays we built in step 1, all shaped `(batch_size, sequence_length)`.
- It returns `last_hidden_state`, shaped `(batch_size, sequence_length, 384)`.

`batch_size` and `sequence_length` are names, not numbers. They're **dynamic** dimensions, so any batch size and any
length up to 256 works.

### 2.2 Run it

`session.run(output_names, input_dict)`. Passing `None` means "give me all outputs".

In [ ]:
feeds = {
    "input_ids":      input_ids,
    "attention_mask": attention_mask,
    "token_type_ids": token_type_ids,
}

outputs = session.run(None, feeds)
token_embeddings = outputs[0]

print("shape:", token_embeddings.shape)
print("first 5 numbers of the [CLS] vector of text 0:", token_embeddings[0, 0, :5])

Read the shape `(2, 13, 384)` as:

> for each of **2** texts, for each of its **13** token positions, a vector of **384** numbers.

So the transformer gives us **one vector per token**, not one per sentence yet. We'll fix that in step 3.

> 🔎 **NumPy note (N.4).** `token_embeddings[0, 0, :5]` = text **0**, token **0** (the `[CLS]` token), the first
> **5** of its 384 numbers. Printing all 2 × 13 × 384 = 9,984 numbers wouldn't tell us much.

> 🔎 **NumPy note (N.2): why we insisted on `int64`.** The model declared its inputs as `tensor(int64)` in 2.1.
> ONNX Runtime doesn't silently convert types. Hand it the wrong dtype and it stops:

In [ ]:
print("output dtype:", token_embeddings.dtype)

try:
    session.run(None, {**feeds, "input_ids": input_ids.astype(np.int32)})
except Exception as e:
    print(type(e).__name__, ":", str(e)[:160])

(PyTorch would often convert for you. With ONNX Runtime, dtypes are part of the model's contract, which is why
`tokenize()` sets `dtype=np.int64` explicitly.)

### 2.3 These are *contextual* vectors

A simple word-vector lookup (like word2vec) gives "python" the same vector every time, whether you mean the
language or the snake. A transformer gives each token a vector that depends on the **whole sentence** around it.
Let's check that:

In [ ]:
def token_vector(sentence, word):
    ids, mask, types = tokenize([sentence])
    out = session.run(None, {"input_ids": ids, "attention_mask": mask, "token_type_ids": types})[0]
    tokens = tokenizer.encode(sentence).tokens
    return out[0, tokens.index(word)]

def cos(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

code_1 = token_vector("I wrote the script in python", "python")
code_2 = token_vector("Python code is easy to read", "python")
snake  = token_vector("A python snake swallowed the rat", "python")

print("python (code) vs python (code) :", round(cos(code_1, code_2), 3))
print("python (code) vs python (snake):", round(cos(code_1, snake), 3))
print()

fruit_1 = token_vector("I ate a sweet red apple", "apple")
fruit_2 = token_vector("She baked an apple pie", "apple")
company = token_vector("Apple released a new iphone", "apple")

print("apple (fruit) vs apple (fruit)  :", round(cos(fruit_1, fruit_2), 3))
print("apple (fruit) vs apple (company):", round(cos(fruit_1, company), 3))

Same token id every time, but different vectors:

- A lookup table would give 1.0 for every pair. Here **none** are 1.0, because each vector depends on its sentence.
- Same meaning (code/code, fruit/fruit) scores clearly **higher** than different meaning (code/snake, fruit/company).

Over 6 layers, the network mixes information from the surrounding words into each token. That's what makes
these embeddings understand that "sign up" and "register" mean the same thing in a search.

### 2.4 Look at the padding positions

Text 1 (`"How to install Docker on Windows?"`) only has 10 real tokens. What did the model output at the 3 padding positions?

In [ ]:
real_len = attention_mask[1].sum()
print("real tokens in text 1:", real_len)
print()
print("vector at a REAL position   :", token_embeddings[1, real_len - 1, :5])
print("vector at a PADDING position:", token_embeddings[1, real_len,     :5])

The padding positions are **not zero**. The model still computes *something* there, and it's meaningless noise.
Remember this for the next step.

### 2.5 What ONNX Runtime does when it loads the file: graph optimization

When we called `ort.InferenceSession(...)`, ONNX Runtime didn't just load the 780 recorded operations. It
**rewrote the graph** first, for example by merging the ~6 small LayerNorm ops into one fused `LayerNormalization`
op, or folding constant calculations ahead of time. This is one of the main reasons it's faster than running the
PyTorch model op by op from Python.

We can ask it to save the graph after optimizing, at each level, and compare:

In [ ]:
import tempfile, time

levels = {
    "DISABLE_ALL ": ort.GraphOptimizationLevel.ORT_DISABLE_ALL,
    "ENABLE_BASIC": ort.GraphOptimizationLevel.ORT_ENABLE_BASIC,
    "ENABLE_ALL  ": ort.GraphOptimizationLevel.ORT_ENABLE_ALL,   # the default
}
bench_ids, bench_mask, bench_types = tokenize(texts * 16)          # a batch of 32
bench_feeds = {"input_ids": bench_ids, "attention_mask": bench_mask, "token_type_ids": bench_types}

original_ops = set(op_counts)
for name, level in levels.items():
    opts = ort.SessionOptions()
    opts.graph_optimization_level = level
    opts.optimized_model_filepath = os.path.join(tempfile.gettempdir(), f"minilm_{name.strip()}.onnx")
    sess = ort.InferenceSession(model_path, opts, providers=["CPUExecutionProvider"])

    sess.run(None, bench_feeds)                                    # warm-up
    start = time.perf_counter()
    for _ in range(10):
        sess.run(None, bench_feeds)
    ms = (time.perf_counter() - start) / 10 * 1000

    optimized = onnx.load(opts.optimized_model_filepath).graph
    new_ops = sorted({n.op_type for n in optimized.node} - original_ops)
    print(f"{name}: {len(optimized.node):4} ops  {ms:6.1f} ms per batch   new fused ops: {new_ops}")

Read it top to bottom: more optimization means **fewer, bigger operations** (the new op types are the fused
replacements). `ENABLE_ALL` fuses LayerNorm and the Gelu activation into single ops and is normally the fastest.
The middle level's timing can be noisy on a laptop, so run the cell twice before reading much into small gaps.
`ENABLE_ALL` is the default, so it's what we've been using all along. You get this for free.

### 2.6 Execution providers: same file, different hardware

The `providers=["CPUExecutionProvider"]` argument picks *which implementation* runs each op. The `.onnx` file
doesn't change, only the provider does:

In [ ]:
print("providers available in this install:", ort.get_available_providers())

| install | provider | runs on |
|---|---|---|
| `onnxruntime` | `CPUExecutionProvider` | any CPU (what we use) |
| `onnxruntime-gpu` | `CUDAExecutionProvider` | NVIDIA GPUs |
| `onnxruntime-directml` | `DmlExecutionProvider` | any DirectX 12 GPU on Windows |
| `onnxruntime` (macOS) | `CoreMLExecutionProvider` | Apple Silicon |

You list providers in order of preference, and ops the first one can't run fall back to the next:
`providers=["CUDAExecutionProvider", "CPUExecutionProvider"]`. With PyTorch the equivalent is
`model.to("cuda")`, but it needs the much bigger CUDA build of torch.

You can also control CPU threads with `opts.intra_op_num_threads = 4`. That's useful in production, where
several worker processes share one machine and shouldn't fight over all the cores.

✅ **Step 2 done.** We have a `(2, 13, 384)` array of token vectors.

---
# Step 3: Mean pooling (many token vectors ➜ one sentence vector)

The config told us `pooling_mode_mean_tokens: true`, so the sentence vector is the **average of its token vectors**.

### 3.1 A toy example by hand

Pretend a text has 3 real tokens and 1 padding token, and vectors have just 2 numbers:

```
token      vector        mask
[CLS]     [1.0, 2.0]      1
hi        [3.0, 4.0]      1
[SEP]     [5.0, 6.0]      1
[PAD]     [9.0, 9.0]      0    ← noise
```

- **Wrong** (average everything): `([1,2]+[3,4]+[5,6]+[9,9]) / 4 = [4.5, 5.25]`
- **Right** (average real tokens only): `([1,2]+[3,4]+[5,6]) / 3 = [3.0, 4.0]`

The trick to do this with arrays is to **multiply by the mask** (which zeroes out padding), sum, and divide by the
**number of real tokens** instead of the total length:

In [ ]:
toy_vectors = np.array([[1.0, 2.0],
                        [3.0, 4.0],
                        [5.0, 6.0],
                        [9.0, 9.0]])
toy_mask = np.array([1, 1, 1, 0])

print("naive mean :", toy_vectors.mean(axis=0))

masked = toy_vectors * toy_mask[:, None]     # [:, None] turns shape (4,) into (4, 1) so it broadcasts over columns
print("masked rows:\n", masked)
print("masked mean:", masked.sum(axis=0) / toy_mask.sum())

### 3.2 Why it matters: same text, different batch

If we got this wrong, the embedding of `"Hi"` would change depending on how long the *other* texts in its batch
are, since longer neighbours mean more padding. Let's prove it:

In [ ]:
def run_model(texts):
    ids, mask, types = tokenize(texts)
    out = session.run(None, {"input_ids": ids, "attention_mask": mask, "token_type_ids": types})[0]
    return out, mask

# "Hi" on its own: no padding at all
alone, alone_mask = run_model(["Hi"])

# "Hi" next to a long text: lots of padding
batched, batched_mask = run_model(["Hi", "This is a much longer sentence that forces a lot of padding onto the short one"])

naive_alone   = alone[0].mean(axis=0)
naive_batched = batched[0].mean(axis=0)
print("NAIVE  mean, 'Hi' alone vs in batch:", round(cos(naive_alone, naive_batched), 4))

The vectors differ, so the naive mean is broken. Now the masked version.

> 🔎 **NumPy note (N.6, N.7): the shapes, step by step.** Masked mean pooling is three NumPy ideas combined.
> Watch the shapes on our real two-text batch before reading the function:

In [ ]:
print("token_embeddings              ", token_embeddings.shape, "  (batch, tokens, 384)")
print("attention_mask                ", attention_mask.shape, "       (batch, tokens)")

# 1. we can't multiply (2, 13, 384) by (2, 13): lined up from the right, 384 vs 13 don't match
try:
    token_embeddings * attention_mask
except ValueError as e:
    print("\n(2,13,384) * (2,13) fails ->", str(e)[:70], "...\n")

# 2. add a size-1 axis: (2, 13) -> (2, 13, 1). Now 384 vs 1 is fine: 1 stretches to 384  (N.7)
m = attention_mask[..., None].astype(np.float32)
print("attention_mask[..., None]     ", m.shape)

# 3. multiply: every number of a padding token's vector becomes 0, real tokens are unchanged
masked = token_embeddings * m
print("token_embeddings * mask       ", masked.shape)
print("   a real token    :", masked[1, 9, :3])
print("   a padding token :", masked[1, 10, :3])

# 4. sum over the tokens (axis 1), divide by how many were real  (N.6)
print("masked.sum(axis=1)            ", masked.sum(axis=1).shape, "      (batch, 384)")
print("m.sum(axis=1)                 ", m.sum(axis=1).shape, "        real-token count per text:", m.sum(axis=1).ravel())

(The `-0.` values are just zero with a sign bit, from multiplying a negative number by 0. They behave exactly like 0.)

`(2, 384) / (2, 1)`: broadcasting again. Each text's 384 sums are divided by that text's own token count.

`np.clip(counts, 1e-9, None)` means "no smaller than 0.000000001, no upper limit". It only matters for an empty
text (count 0), where it avoids a division by zero.

Here's the same thing as a function:

In [ ]:
def mean_pool(token_embeddings, attention_mask):
    # (batch, seq) -> (batch, seq, 1) so it can multiply every one of the 384 numbers
    mask = attention_mask[..., None].astype(np.float32)

    summed = (token_embeddings * mask).sum(axis=1)       # padding rows become 0, then add up -> (batch, 384)
    counts = mask.sum(axis=1)                             # real tokens per text            -> (batch, 1)
    counts = np.clip(counts, 1e-9, None)                  # avoid dividing by 0 (empty text)
    return summed / counts

masked_alone   = mean_pool(alone,   alone_mask)[0]
masked_batched = mean_pool(batched, batched_mask)[0]
print("MASKED mean, 'Hi' alone vs in batch:", round(cos(masked_alone, masked_batched), 4))

Now it's ~1.0: the same text gives the same embedding no matter what it's batched with. ✅

### 3.3 Apply it to our two texts

In [ ]:
sentence_vectors = mean_pool(token_embeddings, attention_mask)

print("before pooling:", token_embeddings.shape, " (batch, tokens, 384)")
print("after pooling: ", sentence_vectors.shape, "     (batch, 384)")

✅ **Step 3 done.** One 384-number vector per text.

---
# Step 4: Normalize (make every vector length 1)

### 4.1 Toy example

The **length** (L2 norm) of a vector is `sqrt(x1² + x2² + …)`. Divide by it, and the vector points the same
way but has length 1.

```
v = [3, 4]      length = sqrt(9 + 16) = 5
v / 5 = [0.6, 0.8]      length = sqrt(0.36 + 0.64) = 1
```

In [ ]:
v = np.array([3.0, 4.0])
print("length:", np.linalg.norm(v))
v_unit = v / np.linalg.norm(v)
print("normalized:", v_unit, " length:", np.linalg.norm(v_unit))

### 4.2 Why bother? So that dot product == cosine similarity

Cosine similarity measures the *angle* between two vectors (1 = same direction, 0 = unrelated):

```
cosine(a, b) = (a · b) / (|a| · |b|)
```

If `|a| = |b| = 1`, the denominator is 1, so **cosine is just the dot product**. That's why vector search code
can simply compute `doc_vectors @ query_vector` and get meaningful scores. Dot products are also the fastest thing
a vector database can compute.

> 🔎 **NumPy note (N.6, N.7): why `keepdims=True`.** `np.linalg.norm(vectors, axis=1)` gives one length per row.
> To divide each row by **its own** length, the lengths must be a column `(2, 1)`, not a flat `(2,)`:

In [ ]:
print("vectors                          ", sentence_vectors.shape)
print("norm(axis=1)                     ", np.linalg.norm(sentence_vectors, axis=1).shape, "   <- (2,) lines up with 384: wrong")
print("norm(axis=1, keepdims=True)      ", np.linalg.norm(sentence_vectors, axis=1, keepdims=True).shape, " <- a column: stretches across 384 ✅")

try:
    sentence_vectors / np.linalg.norm(sentence_vectors, axis=1)
except ValueError as e:
    print("\nwithout keepdims ->", str(e)[:70], "...")

With only 2 texts it fails loudly, which is good. The dangerous case is a **square** array, e.g. 384 texts of 384
numbers: the shapes `(384, 384) / (384,)` *do* broadcast, dividing **columns** instead of rows, and you get wrong
vectors with no error. `keepdims=True` makes the intent explicit.

In [ ]:
def normalize(vectors):
    return vectors / np.linalg.norm(vectors, axis=1, keepdims=True)

lengths_before = np.linalg.norm(sentence_vectors, axis=1)
embeddings     = normalize(sentence_vectors)
lengths_after  = np.linalg.norm(embeddings, axis=1)

print("lengths before:", lengths_before)
print("lengths after: ", lengths_after)

a, b = sentence_vectors
print()
print("cosine (formula) on raw vectors   :", cos(a, b))
print("dot product on normalized vectors :", float(embeddings[0] @ embeddings[1]))

Same number. ✅ **Step 4 done.** `embeddings` is the final result.

---
# Step 5: Did we get it right? Compare with SentenceTransformer

If we copied the recipe correctly, our vectors should match `SentenceTransformer.encode()` up to tiny
floating-point rounding (around 1e-7).

> This step (and the check in 8.4) is the **only** place PyTorch is used. It's optional: if
> `sentence-transformers` isn't installed, the cell says so and everything else still runs.

In [ ]:
try:
    from sentence_transformers import SentenceTransformer
    HAVE_ST = True
except ImportError:
    HAVE_ST = False
    print("sentence-transformers not installed: skipping the comparison (pip install sentence-transformers to run it)")

if HAVE_ST:
    st_model = SentenceTransformer(REPO)
    st_embeddings = st_model.encode(texts)

    print("cosine(ours, theirs) per text:", (embeddings * st_embeddings).sum(axis=1))
    print("largest difference in any number:", np.abs(embeddings - st_embeddings).max())

Cosine of 1.0 and differences around 0.0000001: **our four steps reproduce SentenceTransformer exactly.** 🎉

---
# Step 6: Package it as a class

We collect the four steps into one class with the same `.encode()` method as SentenceTransformer:

- `encode("a string")` returns shape `(384,)`
- `encode(["a", "list"])` returns shape `(n, 384)`

Most search and RAG code only ever calls `embedder.encode(...)`, so this class is a **drop-in replacement**.

We also process big lists in **batches**. Embedding ~1,500 documents in one call would pad
everything to the longest document and use a lot of memory.

In [ ]:
class OnnxEmbedder:
    def __init__(self, repo=REPO, onnx_file="onnx/model.onnx", max_length=256):
        # Step 1 setup: tokenizer with OUR truncation + padding (overrides the file's 128 defaults)
        self.tokenizer = Tokenizer.from_file(hf_hub_download(repo, "tokenizer.json"))
        self.tokenizer.enable_truncation(max_length=max_length)
        self.tokenizer.enable_padding(pad_id=0, pad_token="[PAD]")

        # Step 2 setup: load the ONNX graph
        self.session = ort.InferenceSession(hf_hub_download(repo, onnx_file),
                                            providers=["CPUExecutionProvider"])
        self.input_names = {i.name for i in self.session.get_inputs()}

    def _encode_batch(self, texts):
        # Step 1: tokenize
        encodings = self.tokenizer.encode_batch(texts)
        ids  = np.array([e.ids            for e in encodings], dtype=np.int64)
        mask = np.array([e.attention_mask for e in encodings], dtype=np.int64)
        feeds = {"input_ids": ids, "attention_mask": mask, "token_type_ids": np.zeros_like(ids)}
        feeds = {k: v for k, v in feeds.items() if k in self.input_names}  # some exports have no token_type_ids

        # Step 2: transformer -> (batch, seq, 384)
        token_embeddings = self.session.run(None, feeds)[0]

        # Step 3 + 4: pool -> (batch, 384), then normalize
        return normalize(mean_pool(token_embeddings, mask))

    def encode(self, texts, batch_size=32):
        single = isinstance(texts, str)
        if single:
            texts = [texts]

        chunks = [self._encode_batch(texts[i:i + batch_size])
                  for i in range(0, len(texts), batch_size)]
        vectors = np.vstack(chunks)

        return vectors[0] if single else vectors

> 🔎 **NumPy note (N.4, N.9).** `encode()` cuts the list into slices `texts[0:32]`, `texts[32:64]`, …, runs each
> batch to get a `(32, 384)` block, and `np.vstack` stacks the blocks on top of each other into one `(n, 384)`
> array. For a single string it returns `vectors[0]`, the first row with shape `(384,)`, so `encode("text")` behaves
> like SentenceTransformer's. A tiny demo with 70 texts:

In [ ]:
blocks = [np.zeros((32, 384)), np.zeros((32, 384)), np.zeros((6, 384))]   # 70 texts in batches of 32
print("block shapes:", [b.shape for b in blocks])
print("np.vstack(blocks).shape:", np.vstack(blocks).shape)

### Try it on a few examples

In [ ]:
model = OnnxEmbedder()

q1 = "Can I still join the course after the start date?"
q2 = "How to install Docker on Windows?"
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."

v1, v2, dv = model.encode(q1), model.encode(q2), model.encode(d)

print("shape:", v1.shape)
print("q1 (joining)  vs answer about registering:", round(float(v1 @ dv), 3))
print("q2 (docker)   vs answer about registering:", round(float(v2 @ dv), 3))

The question about joining scores much higher against the registration answer than the Docker question does.
Same behaviour as before, now without PyTorch.

---
# Step 7 (optional): A quantized model, and why you should always measure

The repo also ships `onnx/model_quint8_avx2.onnx`. **Quantization** stores each weight as an 8-bit integer instead
of a 32-bit float:

- the file is **~4× smaller** (23 MB vs 90 MB)
- results are **slightly** different (a small accuracy cost)
- it *can* be faster, but only if your CPU has fast int8 instructions. **It isn't guaranteed.** Let's measure.

We do one warm-up call first, because the first `run()` includes one-time setup cost.

In [ ]:
import time

q_model = OnnxEmbedder(onnx_file="onnx/model_quint8_avx2.onnx")
sample = [q1, q2, d] * 100      # 300 texts

for name, m in [("fp32 ", model), ("int8 ", q_model)]:
    m.encode(sample[:8])                     # warm-up
    start = time.perf_counter()
    m.encode(sample)
    print(f"{name}: {time.perf_counter() - start:.2f} s for {len(sample)} texts")

print()
print("cosine(fp32, int8) on the same texts:", (model.encode([q1, q2, d]) * q_model.encode([q1, q2, d])).sum(axis=1))

The int8 vectors are ~0.99 similar to fp32: close, but **not** the same. Two rules follow from that:

1. **Never mix models.** If you embed documents with fp32 and queries with int8 (or vice versa), scores get worse.
   Whatever model built the index must also embed the queries.
2. **Benchmark on your hardware.** Production teams pick fp32 vs int8 by measuring speed *and* search quality on
   their own data, not by assuming.

---
# Step 8: Embed the documents AND the queries

A vector search has **two** embedding moments, and both must use the same embedder:

```
 OFFLINE (once)                          ONLINE (every user question)
 ──────────────                          ────────────────────────────
 FAQ docs ─► model.encode(docs) ─► index    query ─► model.encode(query) ─► search index ─► top-k docs
```

So far we've only encoded a few hand-picked sentences. Now we do the real thing: embed **every FAQ document** with
`OnnxEmbedder`, build an index from those vectors, and search it with ONNX-encoded queries. No SentenceTransformer anywhere.

### 8.1 Load the documents

We use the public FAQ of the free [DataTalks.Club](https://datatalks.club) courses: about 1,500 question/answer
pairs, published as JSON. One file lists the courses, and each course has its own file of FAQ entries.

We use only the standard library (`urllib`) to download it.

In [ ]:
import urllib.request

FAQ_BASE = "https://datatalksclub.github.io/faq"

def fetch_json(url):
    with urllib.request.urlopen(url, timeout=30) as response:
        return json.load(response)

documents = []
for course in fetch_json(f"{FAQ_BASE}/json/courses.json"):
    documents.extend(fetch_json(f"{FAQ_BASE}{course['path']}"))

print(len(documents), "documents")
print("fields:", list(documents[0].keys()))

Each document has a `question`, an `answer` and the `course` it belongs to. We embed the question and answer
**together** as one text, so a search can match on either:

In [ ]:
doc_texts = [f"{doc['question']} {doc['answer']}" for doc in documents]
print(doc_texts[0][:300])

> Whatever text format you choose here, keep it fixed. If you later embed documents in a different format, the
> new vectors won't be comparable with the old ones.

### 8.2 Embed all documents (the offline step)

`encode()` already splits the list into batches of 32 for us. Documents are much longer than queries (many hit the
256-token limit), so this is the slow part. In production it runs as a batch job, not per request.

In [ ]:
start = time.perf_counter()
doc_vectors = model.encode(doc_texts, batch_size=32)
print(f"embedded {len(doc_texts)} documents in {time.perf_counter() - start:.1f} s")
print("shape:", doc_vectors.shape, " <- one 384-d vector per document")

How many documents were too long and got truncated? Their endings don't influence their vectors at all:

In [ ]:
# count REAL tokens (sum of the attention mask), not len(e.ids):
# padding stretches every row in the batch to the longest one, so len(e.ids) would always be 256 here
lengths = [sum(e.attention_mask) for e in model.tokenizer.encode_batch(doc_texts)]
truncated = sum(n >= 256 for n in lengths)
print(f"{truncated} of {len(lengths)} documents hit the 256-token limit ({truncated / len(lengths):.0%})")
print(f"median document length: {int(np.median(lengths))} tokens")

### 8.3 Vector search from scratch (the online step)

A "vector index" sounds fancy, but for a few thousand documents it's just the matrix `doc_vectors` plus three lines
of NumPy: score every document with one dot product, then take the highest scores.

In [ ]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)                     # same embedder as the documents

scores = doc_vectors @ query_vector                    # (n_docs,), cosine similarity because everything is normalized
top = np.argsort(-scores)[:5]
for i in top:
    print(f"{scores[i]:.3f}  [{documents[i]['course']}]  {documents[i]['question']}")

> 🔎 **NumPy note (N.8, N.9).**
> - `doc_vectors @ query_vector` is `(n_docs, 384) @ (384,) -> (n_docs,)`: one dot product per document, in one line.
> - `np.argsort(-scores)` gives document **positions** ordered from best to worst. We negate because `argsort`
>   sorts smallest-first. `[:5]` keeps the first five positions.
> - We then use those positions to look up `documents[i]`. That's why we want positions (`argsort`) rather than the
>   sorted scores themselves (`np.sort`).

The same question exists in several courses' FAQs. Real search systems let you **filter** on metadata. With
NumPy, a filter is a **boolean mask**: an array of `True`/`False`, one per document.

In [ ]:
doc_courses = np.array([doc["course"] for doc in documents])

def search(query, k=5, course=None):
    scores = doc_vectors @ model.encode(query)
    if course is not None:
        allowed = doc_courses == course                 # boolean array, True where the course matches
        scores = np.where(allowed, scores, -np.inf)     # filtered-out docs get -infinity, so they sort last
    top = np.argsort(-scores)[:k]
    return [(float(scores[i]), documents[i]) for i in top]

print("courses:", sorted({doc["course"] for doc in documents}))
print()
for score, doc in search("I just discovered the course. Can I still join it?", course="llm-zoomcamp"):
    print(f"{score:.3f}  {doc['question']}")

> 🔎 **NumPy note: boolean masks.** `doc_courses == course` compares *every* element at once and returns an array of
> `True`/`False`. `np.where(condition, a, b)` picks from `a` where the condition is `True` and from `b` where it's
> `False`. Here, it keeps the real score or replaces it with `-inf`.

This brute-force search checks every document, so it's always exact. Vector databases (pgvector, Qdrant, FAISS, …)
add **approximate** indexes like HNSW or IVF to skip most documents when there are millions of them, trading a little
accuracy for a lot of speed.

### 8.4 Would SentenceTransformer have found the same documents?

If our pipeline is truly equivalent, embedding the documents with SentenceTransformer should give the same search
results. To keep it quick, we compare on the first 300 documents and a few different queries (optional: needs
`sentence-transformers`).

In [ ]:
if HAVE_ST:
    subset = doc_texts[:300]
    ours   = model.encode(subset)
    theirs = st_model.encode(subset)
    print("largest difference in any document vector:", np.abs(ours - theirs).max())
    print()

    for q in ["Can I still join after the start?", "How do I get a certificate?", "Docker doesn't work on Windows"]:
        top_ours   = np.argsort(-(ours   @ model.encode(q)))[:5]
        top_theirs = np.argsort(-(theirs @ st_model.encode(q)))[:5]
        print(f"{q!r:40} same top-5: {list(top_ours) == list(top_theirs)}")
else:
    print("sentence-transformers not installed: skipping")

Same documents, same order. That's what makes swapping embedders safe *here*. But the **general rule** is:
whatever embedder serves queries must also be the one that built the index. When you switch models (or switch to
the int8 version), re-run 8.2 and rebuild the index.

> **Side note: asymmetric models.** MiniLM treats queries and documents the same way. Some models (e5, BGE, nomic)
> are trained with **different prefixes**, e.g. `"query: ..."` vs `"passage: ..."`. With those, forgetting the
> prefix on one side quietly hurts search quality. Always check the model card.

### 8.5 From search to RAG

RAG (retrieval-augmented generation) is this search plus an LLM: retrieve the most relevant documents, paste them
into the prompt, and ask the LLM to answer using only that context. The embedder's job ends at retrieval:

In [ ]:
def build_prompt(question, k=3, course="llm-zoomcamp"):
    results = search(question, k, course)
    context = "\n\n".join(f"Q: {doc['question']}\nA: {doc['answer']}" for _, doc in results)
    return (
        "Answer the question using only the FAQ entries below.\n\n"
        f"FAQ:\n{context}\n\n"
        f"Question: {question}"
    )

print(build_prompt("the program has already begun, can I still sign up?")[:1200])

Send that prompt to any LLM (a hosted API, Ollama, llama.cpp, …) and you have a working RAG system.

The embedder runs **once per question** here (online). The documents were embedded **once, up front** in 8.2 (offline).

---
# Step 9: Replacing sentence-transformers + PyTorch in your own code

Everything above was the "how". This step is the practical swap.

### 9.1 Before and after

Only the line that creates the model changes. Everything that calls `.encode()` stays the same:

```python
# BEFORE: needs sentence-transformers + torch (~580 MB installed)
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# AFTER: needs onnxruntime + tokenizers + numpy (~60 MB installed)
from embedder import OnnxEmbedder
model = OnnxEmbedder("sentence-transformers/all-MiniLM-L6-v2")

# unchanged, works with either:
doc_vectors  = model.encode(doc_texts)               # documents (offline)
query_vector = model.encode("Can I still join?")     # queries (online)
scores = doc_vectors @ query_vector
```

This works because of **duck typing**: code that calls `model.encode(...)` never checks *what* `model` is, only
that it has an `.encode()` that returns the right vectors.

### 9.2 Move the class into a module

To use it outside this notebook, copy `mean_pool`, `normalize` and the `OnnxEmbedder` class into an
`embedder.py` in your project. They're self-contained (imports: `numpy`, `onnxruntime`, `tokenizers`,
`huggingface_hub`).

### 9.3 Dependencies

What production needs:

```bash
pip install onnxruntime tokenizers huggingface-hub numpy
```

What's only useful while developing (verifying against the original, inspecting graphs):

```bash
pip install sentence-transformers onnx
```

Keep these two lists separate (e.g. `requirements.txt` vs `requirements-dev.txt`, or `uv add --dev ...`), so
your production install and server image never contain PyTorch.

### 9.4 What if a model has no `onnx/` folder?

MiniLM came pre-exported. For a model that doesn't, you export it yourself, **once, on your dev machine** (this is
the only time PyTorch is needed):

**The easy way:** Hugging Face's `optimum` does the export and also writes the tokenizer files:

```bash
uvx --from "optimum[exporters]" optimum-cli export onnx --model BAAI/bge-small-en-v1.5 bge_onnx/
```

**The manual way**, to see what it does. This is the `torch.onnx.export` from the diagram at the start of Step 2:

```python
import torch
from transformers import AutoModel, AutoTokenizer

name = "sentence-transformers/all-MiniLM-L6-v2"
tok = AutoTokenizer.from_pretrained(name)
net = AutoModel.from_pretrained(name).eval()

example = tok(["hello world"], return_tensors="pt")      # any input works; it's only used to trace the ops

torch.onnx.export(
    net,
    (example["input_ids"], example["attention_mask"], example["token_type_ids"]),
    "model.onnx",
    input_names=["input_ids", "attention_mask", "token_type_ids"],
    output_names=["last_hidden_state"],
    # without this, the file would ONLY accept batch=1, length=4 (the example's shape).
    # this is where the 'batch_size' / 'sequence_length' names we saw in 2.1 come from.
    dynamic_axes={n: {0: "batch_size", 1: "sequence_length"}
                  for n in ["input_ids", "attention_mask", "token_type_ids", "last_hidden_state"]},
    opset_version=14,
)
```

Notice the export only covers **the network** (step 2). Tokenizing, pooling and normalizing aren't in the file.
That's why we wrote them ourselves, and why they're where most bugs hide.

### 9.5 Checklist when you switch to another model

Each item is something this notebook ran into:

1. **Tokenizer settings**: check `tokenizer.truncation` / `tokenizer.padding` and set the max length from
   `sentence_bert_config.json` (the 128-vs-256 gotcha in 1.0).
2. **Pooling**: read `1_Pooling/config.json`. Mean (MiniLM), CLS (BGE) or something else.
3. **Normalize?** Check `modules.json` for a `Normalize` step.
4. **Inputs**: some exports have no `token_type_ids` (our class already handles that).
5. **Prefixes**: some models need `"query: "` / `"passage: "` (see 8.4).
6. **Verify** against SentenceTransformer, as in Step 5, before trusting it.
7. **Re-embed the documents** and rebuild the index with the new model.

### 9.6 What you gain (example numbers: one 14-core Windows laptop, CPU only, same model)

| | sentence-transformers + PyTorch | ONNX Runtime |
|---|---|---|
| installed size | ~580 MB (torch 517 + transformers 63) | ~50 MB |
| cold start (import + load) | ~19–21 s | ~3 s |
| memory once loaded | ~600 MB | ~250 MB |
| one query (median) | ~13–15 ms | ~6–8 ms |
| ~1,500 FAQ docs (avg of 2 runs) | ~63 s | ~50–55 s |
| vectors | – | identical (max difference ~2e-7) |

The big wins are size, startup and memory, which matter most when you deploy. Bulk embedding is only modestly faster,
because both run the same maths with optimized CPU kernels. Your numbers will differ. Laptop timings in particular
swing ±50% with heat and background load, so compare runs made back to back, not across sessions.

---
# Recap

| step | input → output | shape | key idea |
|---|---|---|---|
| 1. tokenize | text → ids + mask | `(batch, seq)` | WordPiece, `[CLS]`/`[SEP]`, truncate at 256, pad + mask |
| 2. transformer | ids → token vectors | `(batch, seq, 384)` | the only trained part; vectors are contextual |
| 3. mean pool | token vectors → sentence vector | `(batch, 384)` | average **real** tokens only (use the mask) |
| 4. normalize | vector → unit vector | `(batch, 384)` | makes dot product == cosine similarity |
| 8. search | docs encoded offline, query encoded online | `(n_docs, 384)` · `(384,)` | **same embedder on both sides**; rebuild the index when the model changes |

**What we dropped:** PyTorch and sentence-transformers at inference time. The runtime is now `onnxruntime` + `tokenizers` + `numpy`.

## Exercises to check your understanding

1. In `mean_pool`, replace the masked mean with `token_embeddings.mean(axis=1)` and re-run step 5. How far off is it from SentenceTransformer, and why?
2. Remove the `normalize(...)` call. Do the search results in step 8 change? Do the scores?
3. Try **CLS pooling** (`token_embeddings[:, 0]`, the `[CLS]` vector only) instead of mean pooling. How does it compare? (Some models, like BGE, are trained for CLS pooling, which is why the pooling config matters.)
4. Embed a FAQ answer that's longer than 256 tokens. Change the last sentence and see if the embedding changes at all.